## Create metadata files (with organized IDs) for the CWRU, PU and UORED-VAFCLS datasets

In this notebook, we process raw files from all datasets, adding important metadata, labels, etc. in order to use it as Pydantic datasets later. We also rename raw data files and save them as .bz2 so each file correspond to a waveform_id, a notation we use to distinct unique waveforms. All metadata dataframes are already available on `data/metadata`, but it is necessary to run this notebook in order to process and save the dataset raw files with our format.

Later, raw signals will be accesible through functions in `src/utils/add_signal_data.py`, which load signals from CWRU, PU and UORED-VAFCLS. Examples of their usage are provided in this notebook.

In [1]:
#imports
import os
import pandas as pd
import autorootcwd #noqa
import numpy as np
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import pandas as pd
from scipy.io import loadmat
from typing import List,Dict
from pathlib import Path

## 1. CWRU dataset 

In [2]:
dir_to_download= "data/datasets/cwru/raw"

#### 1.1 Downloading CWRU waveforms:

In [ ]:
# Download CWRU dataset if not already present: 

def download_cwru_dataset():
    # The four main pages containing the dataset links
    base_urls = [
        "https://engineering.case.edu/bearingdatacenter/normal-baseline-data",
        "https://engineering.case.edu/bearingdatacenter/12k-drive-end-bearing-fault-data",
        "https://engineering.case.edu/bearingdatacenter/48k-drive-end-bearing-fault-data",
        "https://engineering.case.edu/bearingdatacenter/12k-fan-end-bearing-fault-data"
    ]
    
    # Create the target directory
    target_dir = dir_to_download
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)
        print(f"Created directory: {target_dir}")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    downloaded_files = set()

    def dl(page_url, headers):

        response = requests.get(page_url, headers=headers)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Find all links ending in .mat
        links = soup.find_all('a', href=True)
        mat_links = [l['href'] for l in links if l['href'].endswith('.mat')]
        
        if not mat_links:
            print(f"  No .mat files found on this page.")

        def dl_file(href):
            full_url = urljoin(page_url, href)
            filename = os.path.basename(full_url)
        
            if filename in downloaded_files:
                return
            
            save_path = os.path.join(target_dir, filename)
            print(f"  Downloading {filename}...")
            file_res = requests.get(full_url, headers=headers, stream=True)
            file_res.raise_for_status()
            
            with open(save_path, 'wb') as f:
                for chunk in file_res.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            downloaded_files.add(filename)

        for href in mat_links:
            try:
                dl_file(href)
            except Exception as e:
                print(f"    Error downloading {href}: {e}")
                print("Retrying in 5 seconds...")
                
                time.sleep(5)
                try:
                    dl_file(href)
                except Exception as e:
                    print(f"    Failed again downloading {href}: {e}")
                    print("Retrying in 10 seconds...")
                    time.sleep(10)
                    try:
                        dl_file(href)
                    except Exception as e:
                        print(f"    Failed again downloading {href}: {e}")
                        print("Skipping this file.")

            time.sleep(2)  # Be polite to the server :)
    for page_url in base_urls:
        print(f"\nScanning page: {page_url}")
        try:
            dl(page_url, headers)
                
        except Exception as e:
            print(f"  Error processing {page_url}: {e}")
            continue

    print(f"\nDownload complete! {len(downloaded_files)} files saved to {target_dir}.")

if __name__ == "__main__":
    download_cwru_dataset()


Scanning page: https://engineering.case.edu/bearingdatacenter/48k-drive-end-bearing-fault-data
    Error downloading https://engineering.case.edu/sites/default/files/123.mat: ('Connection broken: IncompleteRead(3638146 bytes read, 4160334 more expected)', IncompleteRead(3638146 bytes read, 4160334 more expected))
Retrying in 5 seconds...
    Error downloading https://engineering.case.edu/sites/default/files/162.mat: ('Connection broken: IncompleteRead(4584979 bytes read, 3139229 more expected)', IncompleteRead(4584979 bytes read, 3139229 more expected))
Retrying in 5 seconds...

Scanning page: https://engineering.case.edu/bearingdatacenter/12k-fan-end-bearing-fault-data

Download complete! 97 files saved to data/datasets/cwru/raw.


#### 1.2 Processing files and organizing metadata

If you already have downloaded all files, store them in `dir_to_download = data/datasets/cwru/raw`

In [3]:
# check all files inside subfolders of path where you downloaded the dataset

all_files = []
count = 0
for root, dirs, files in os.walk(dir_to_download):
    for file in files:
        if file.endswith(".mat"):
            all_files.append(os.path.join(root, file))
            count += 1

In [4]:
import scipy.io as sio
import re
from pathlib import Path
from collections import defaultdict

def extract_multi_group(mat_dict):
    """
    Extracts all X### groups from a single .mat dictionary.
    Example keys it handles:
        X215_DE_time, X215_FE_time, X215RPM, X217_DE_time, ...
    Returns:
        {"X215": {...}, "X217": {...}}
    """

    # Create dict of dicts with default None
    groups = defaultdict(lambda: {"DE": None, "FE": None, "BA": None, "RPM": None})

    for key, value in mat_dict.items():
        if key in ["__header__", "__version__", "__globals__"]:
            continue

        # Match X###_DE_time, X###_FE_time, X###_BA_time
        m = re.match(r"(X\d+)_(DE|FE|BA)_time$", key)
        if m:
            group = m.group(1)    # X215
            channel = m.group(2)  # DE/FE/BA
            groups[group][channel] = value.squeeze()
            continue

        # Match RPM key: X###RPM
        m = re.match(r"(X\d+)RPM$", key)
        if m:
            group = m.group(1)
            groups[group]["RPM"] = value.squeeze()
            continue

    return dict(groups)


def load_all_mat_files(folder):
    folder = Path(folder)
    dataset = {}

    for file in folder.glob("*.mat"):
        try:
            mat = sio.loadmat(file)
        except Exception as e:
            print(f"Could not load {file}: {e}")
            continue

        groups = extract_multi_group(mat)

        # Insert each group into dataset
        for gname, gdata in groups.items():
            dataset[gname] = gdata

    return dataset


# -------- USAGE ----------
folder = dir_to_download
dataset = load_all_mat_files(folder)

dataset

{'X110': {'DE': array([ 0.03212677,  0.07823077,  0.16146831, ..., -0.08845292,
         -0.02899754,  0.00855323], shape=(486224,)),
  'FE': array([0.08629091, 0.11053455, 0.11012364, ..., 0.40145818, 0.46782   ,
         0.52534727], shape=(486224,)),
  'BA': None,
  'RPM': array(1772, dtype=uint16)},
 'X119': {'DE': array([-0.0721212 ,  0.30310395,  0.03313677, ...,  0.08820228,
         -0.19687138, -0.03898443], shape=(121410,)),
  'FE': array([ 0.07745636,  0.04561091, -0.01109455, ..., -0.01273818,
          0.04170727,  0.02753091], shape=(121410,)),
  'BA': array([0.07327228, 0.06932902, 0.05480332, ..., 0.05753947, 0.01569258,
         0.01488783], shape=(121410,)),
  'RPM': array(1772, dtype=uint16)},
 'X132': {'DE': array([-0.02964441,  0.11938982,  0.06253752, ...,  0.6538014 ,
         -0.35045379, -1.07856926], shape=(121410,)),
  'FE': array([ 0.07190909,  0.36591455,  0.30694909, ..., -0.18655273,
          0.20935818, -0.07047091], shape=(121410,)),
  'BA': array([-0.

In [5]:
def extract_prefix(k):
    if "_" in k:
        return k.split("_")[0]
    m = re.match(r"(X\d+)", k)
    return m.group(1) if m else k

def load_cwru_file_to_df(file_path):
    data = sio.loadmat(file_path)
    keys = [k for k in data.keys() if not k.startswith("__")]

    rows = []
    file_id = os.path.splitext(os.path.basename(file_path))[0].split('_')[0]
    file_id = file_id.replace("@6", "").replace("@12", "").replace("@3", "")

    # Group keys by prefix
    prefixes = {}
    for k in keys:
        prefix = extract_prefix(k)
        prefixes.setdefault(prefix, []).append(k)

    for prefix, klist in prefixes.items():

        # Find RPM key
        rpm_key = next((k for k in klist if "RPM" in k.upper()), None)
        rpm_value = float(data[rpm_key].squeeze()) if rpm_key else None

        # Process signal keys
        for k in klist:
            if "time" not in k.lower():
                continue

            signal = data[k].squeeze()

            rows.append({
                "file": file_id,
                "prefix": prefix,
                "signal_name": k,
                "rpm": rpm_value,
                "signal": signal
            })

    return pd.DataFrame(rows)



df = pd.DataFrame()
for file in all_files:
    df = pd.concat([df,load_cwru_file_to_df(file)],ignore_index=True)

/tmp/ipykernel_252843/1937484556.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,load_cwru_file_to_df(file)],ignore_index=True)


In [6]:
df # Dataframe contains file, prefix (id contained inside the file), signal_name, rpm, signal

,file,prefix,signal_name,rpm,signal
0,110,X110,X110_DE_time,1772.0,"[0.03212676923076923, 0.07823076923076923, 0.1..."
1,110,X110,X110_FE_time,1772.0,"[0.08629090909090908, 0.11053454545454545, 0.1..."
2,119,X119,X119_DE_time,1772.0,"[-0.07212119760479042, 0.30310395209580837, 0...."
3,119,X119,X119_FE_time,1772.0,"[0.07745636363636363, 0.045610909090909084, -0..."
4,119,X119,X119_BA_time,1772.0,"[0.07327228486646885, 0.06932902077151336, 0.0..."
...,...,...,...,...,...
411,213,X213,X213_DE_time,1796.0,"[0.851568, 0.6287667692307692, 0.3158436923076..."
412,213,X213,X213_FE_time,1796.0,"[0.8386654545454545, 0.61924, 0.31105818181818..."
413,291,X291,X291_DE_time,1775.0,"[0.0320809381237525, 0.20141956087824353, 0.03..."
414,291,X291,X291_FE_time,1775.0,"[-0.16300389610389612, 0.01625974025974026, 0...."


In [7]:
### Mapping dictionary for each file based on its webpage description: 

all_metadata = {
    #/* Normal Baseline Data */
    "97": {"fault_diameter": "Normal", "load": "0", "rpm": "1797", "label": "Normal_0"},
    "98": {"fault_diameter": "Normal", "load": "1", "rpm": "1772", "label": "Normal_1"},
    "99": {"fault_diameter": "Normal", "load": "2", "rpm": "1750", "label": "Normal_2"},
    "100": {"fault_diameter": "Normal", "load": "3", "rpm": "1730", "label": "Normal_3"},

    #/* 12k Drive End Bearing Fault Data */
    "105": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "IR007_0"},
    "118": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "B007_0"},
    "130": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "OR007@6_0"},
    "144": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "OR007@3_0"},
    "156": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "OR007@12_0"},

    "106": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "IR007_1"},
    "119": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "B007_1"},
    "131": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "OR007@6_1"},
    "145": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "OR007@3_1"},
    "158": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "OR007@12_1"},

    "107": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "IR007_2"},
    "120": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "B007_2"},
    "132": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "OR007@6_2"},
    "146": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "OR007@3_2"},
    "159": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "OR007@12_2"},

    "108": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "IR007_3"},
    "121": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "B007_3"},
    "133": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "OR007@6_3"},
    "147": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "OR007@3_3"},
    "160": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "OR007@12_3"},

    "169": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "IR014_0"},
    "185": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "B014_0"},
    "197": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "OR014@6_0"},

    "170": {"fault_diameter": "0.014\"", "load": "1", "rpm": "1772", "label": "IR014_1"},
    "186": {"fault_diameter": "0.014\"", "load": "1", "rpm": "1772", "label": "B014_1"},
    "198": {"fault_diameter": "0.014\"", "load": "1", "rpm": "1772", "label": "OR014@6_1"},

    "171": {"fault_diameter": "0.014\"", "load": "2", "rpm": "1750", "label": "IR014_2"},
    "187": {"fault_diameter": "0.014\"", "load": "2", "rpm": "1750", "label": "B014_2"},
    "199": {"fault_diameter": "0.014\"", "load": "2", "rpm": "1750", "label": "OR014@6_2"},

    "172": {"fault_diameter": "0.014\"", "load": "3", "rpm": "1730", "label": "IR014_3"},
    "188": {"fault_diameter": "0.014\"", "load": "3", "rpm": "1730", "label": "B014_3"},
    "200": {"fault_diameter": "0.014\"", "load": "3", "rpm": "1730", "label": "OR014@6_3"},

    "209": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "IR021_0"},
    "222": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "B021_0"},
    "234": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "OR021@6_0"},
    "246": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "OR021@3_0"},
    "258": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "OR021@12_0"},

    "210": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "IR021_1"},
    "223": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "B021_1"},
    "235": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "OR021@6_1"},
    "247": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "OR021@3_1"},
    "259": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "OR021@12_1"},

    "211": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "IR021_2"},
    "224": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "B021_2"},
    "236": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "OR021@6_2"},
    "248": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "OR021@3_2"},
    "260": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "OR021@12_2"},

    "212": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "IR021_3"},
    "225": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "B021_3"},
    "237": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "OR021@6_3"},
    "249": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "OR021@3_3"},
    "261": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "OR021@12_3"},

    "3001": {"fault_diameter": "0.028\"", "load": "0", "rpm": "1797", "label": "IR028_0"},
    "3005": {"fault_diameter": "0.028\"", "load": "0", "rpm": "1797", "label": "B028_0"},

    "3002": {"fault_diameter": "0.028\"", "load": "1", "rpm": "1772", "label": "IR028_1"},
    "3006": {"fault_diameter": "0.028\"", "load": "1", "rpm": "1772", "label": "B028_1"},

    "3003": {"fault_diameter": "0.028\"", "load": "2", "rpm": "1750", "label": "IR028_2"},
    "3007": {"fault_diameter": "0.028\"", "load": "2", "rpm": "1750", "label": "B028_2"},

    "3004": {"fault_diameter": "0.028\"", "load": "3", "rpm": "1730", "label": "IR028_3"},
    "3008": {"fault_diameter": "0.028\"", "load": "3", "rpm": "1730", "label": "B028_3"},

    #/* 12k Fan End Bearing Fault Data (Apenas dados de 0.007" e 0.014" e 0.021" são fornecidos nesta página) */
    "278": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "IR007_0_FE"},
    "282": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "B007_0_FE"},
    "294": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "OR007@6_0_FE"},
    "298": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "OR007@3_0_FE"},
    "302": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "OR007@12_0_FE"},

    "279": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "IR007_1_FE"},
    "283": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "B007_1_FE"},
    "295": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "OR007@6_1_FE"},
    "299": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "OR007@3_1_FE"},
    "305": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "OR007@12_1_FE"},

    "280": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "IR007_2_FE"},
    "284": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "B007_2_FE"},
    "296": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "OR007@6_2_FE"},
    "300": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "OR007@3_2_FE"},
    "306": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "OR007@12_2_FE"},

    "281": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "IR007_3_FE"},
    "285": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "B007_3_FE"},
    "297": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "OR007@6_3_FE"},
    "301": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "OR007@3_3_FE"},
    "307": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "OR007@12_3_FE"},

    "274": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "IR014_0_FE"},
    "286": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "B014_0_FE"},
    "313": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "OR014@6_0_FE"},
    "310": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "OR014@3_0_FE"},

    "275": {"fault_diameter": "0.014\"", "load": "1", "rpm": "1772", "label": "IR014_1_FE"},
    "287": {"fault_diameter": "0.014\"", "load": "1", "rpm": "1772", "label": "B014_1_FE"},
    "309": {"fault_diameter": "0.014\"", "load": "1", "rpm": "1772", "label": "OR014@3_1_FE"},

    "276": {"fault_diameter": "0.014\"", "load": "2", "rpm": "1750", "label": "IR014_2_FE"},
    "288": {"fault_diameter": "0.014\"", "load": "2", "rpm": "1750", "label": "B014_2_FE"},
    "311": {"fault_diameter": "0.014\"", "load": "2", "rpm": "1750", "label": "OR014@3_2_FE"},

    "277": {"fault_diameter": "0.014\"", "load": "3", "rpm": "1730", "label": "IR014_3_FE"},
    "289": {"fault_diameter": "0.014\"", "load": "3", "rpm": "1730", "label": "B014_3_FE"},
    "312": {"fault_diameter": "0.014\"", "load": "3", "rpm": "1730", "label": "OR014@3_3_FE"},

    "270": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "IR021_0_FE"},
    "290": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "B021_0_FE"},
    "315": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "OR021@6_0_FE"},

    "271": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "IR021_1_FE"},
    "291": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "B021_1_FE"},
    "316": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "OR021@3_1_FE"},

    "272": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "IR021_2_FE"},
    "292": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "B021_2_FE"},
    "317": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "OR021@3_2_FE"},

    "273": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "IR021_3_FE"},
    "293": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "B021_3_FE"},
    "318": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "OR021@3_3_FE"},

    #/* 48k Drive End Bearing Fault Data */
    "109": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "IR007_0_48k"},
    "122": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "B007_0_48k"},
    "135": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "OR007@6_0_48k"},
    "148": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "OR007@3_0_48k"},
    "161": {"fault_diameter": "0.007\"", "load": "0", "rpm": "1797", "label": "OR007@12_0_48k"},

    "110": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "IR007_1_48k"},
    "123": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "B007_1_48k"},
    "136": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "OR007@6_1_48k"},
    "149": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "OR007@3_1_48k"},
    "162": {"fault_diameter": "0.007\"", "load": "1", "rpm": "1772", "label": "OR007@12_1_48k"},

    "111": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "IR007_2_48k"},
    "124": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "B007_2_48k"},
    "137": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "OR007@6_2_48k"},
    "150": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "OR007@3_2_48k"},
    "163": {"fault_diameter": "0.007\"", "load": "2", "rpm": "1750", "label": "OR007@12_2_48k"},

    "112": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "IR007_3_48k"},
    "125": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "B007_3_48k"},
    "138": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "OR007@6_3_48k"},
    "151": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "OR007@3_3_48k"},
    "164": {"fault_diameter": "0.007\"", "load": "3", "rpm": "1730", "label": "OR007@12_3_48k"},

    "174": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "IR014_0_48k"},
    "189": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "B014_0_48k"},
    "201": {"fault_diameter": "0.014\"", "load": "0", "rpm": "1797", "label": "OR014@6_0_48k"},

    "175": {"fault_diameter": "0.014\"", "load": "1", "rpm": "1772", "label": "IR014_1_48k"},
    "190": {"fault_diameter": "0.014\"", "load": "1", "rpm": "1772", "label": "B014_1_48k"},
    "202": {"fault_diameter": "0.014\"", "load": "1", "rpm": "1772", "label": "OR014@6_1_48k"},

    "176": {"fault_diameter": "0.014\"", "load": "2", "rpm": "1750", "label": "IR014_2_48k"},
    "191": {"fault_diameter": "0.014\"", "load": "2", "rpm": "1750", "label": "B014_2_48k"},
    "203": {"fault_diameter": "0.014\"", "load": "2", "rpm": "1750", "label": "OR014@6_2_48k"},

    "177": {"fault_diameter": "0.014\"", "load": "3", "rpm": "1730", "label": "IR014_3_48k"},
    "192": {"fault_diameter": "0.014\"", "load": "3", "rpm": "1730", "label": "B014_3_48k"},
    "204": {"fault_diameter": "0.014\"", "load": "3", "rpm": "1730", "label": "OR014@6_3_48k"},

    "213": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "IR021_0_48k"},
    "226": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "B021_0_48k"},
    "238": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "OR021@6_0_48k"},
    "250": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "OR021@3_0_48k"},
    "262": {"fault_diameter": "0.021\"", "load": "0", "rpm": "1797", "label": "OR021@12_0_48k"},

    "214": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "IR021_1_48k"},
    "227": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "B021_1_48k"},
    "239": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "OR021@6_1_48k"},
    "251": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "OR021@3_1_48k"},
    "263": {"fault_diameter": "0.021\"", "load": "1", "rpm": "1772", "label": "OR021@12_1_48k"},

    "215": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "IR021_2_48k"},
    "228": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "B021_2_48k"},
    "240": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "OR021@6_2_48k"},
    "252": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "OR021@3_2_48k"},
    "264": {"fault_diameter": "0.021\"", "load": "2", "rpm": "1750", "label": "OR021@12_2_48k"},

    "217": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "IR021_3_48k"},
    "229": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "B021_3_48k"},
    "241": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "OR021@6_3_48k"},
    "253": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "OR021@3_3_48k"},
    "265": {"fault_diameter": "0.021\"", "load": "3", "rpm": "1730", "label": "OR021@12_3_48k"}
}

In [8]:
for key in all_metadata.keys():
    df.loc[(df['file'] == key), 'fault_metadata'] = all_metadata[key]['label']
    df.loc[(df['file'] == key), 'fault_size'] = all_metadata[key]['fault_diameter']
    df.loc[(df['file'] == key), 'load'] = all_metadata[key]['load']
    df.loc[(df['file'] == key), 'rpm'] = all_metadata[key]['rpm']

/tmp/ipykernel_252843/1779656119.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1797' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[(df['file'] == key), 'rpm'] = all_metadata[key]['rpm']


In [9]:
df.head()

,file,prefix,signal_name,rpm,signal,fault_metadata,fault_size,load
0,110,X110,X110_DE_time,1772,"[0.03212676923076923, 0.07823076923076923, 0.1...",IR007_1_48k,"0.007""",1
1,110,X110,X110_FE_time,1772,"[0.08629090909090908, 0.11053454545454545, 0.1...",IR007_1_48k,"0.007""",1
2,119,X119,X119_DE_time,1772,"[-0.07212119760479042, 0.30310395209580837, 0....",B007_1,"0.007""",1
3,119,X119,X119_FE_time,1772,"[0.07745636363636363, 0.045610909090909084, -0...",B007_1,"0.007""",1
4,119,X119,X119_BA_time,1772,"[0.07327228486646885, 0.06932902077151336, 0.0...",B007_1,"0.007""",1


In [10]:
df['fault_type'] = df.fault_metadata.apply(lambda x: x.split('_')[0].replace("007", "").replace("021", "").replace("014","").replace("028", ""))
df['fault_location'] = "DE"
df.loc[df.fault_metadata.apply(lambda x: "FE" in x), 'fault_location'] = 'FE'
df['signal_location'] = df.signal_name.apply(lambda x: x.split('_')[1])
df['fs'] = 12000
df.loc[df.fault_metadata.apply(lambda x: "48k" in x), 'fs'] = 48000
df.fault_size = df.fault_size.apply(lambda x: float(x.replace('"', '').replace("Normal", "0"))*1000).astype(int)

# Bearing characteristic frequencies
df.loc[df.signal_location == "DE", ['bpfo', 'bpfi', 'bsf', 'ftf']] = [3.5848,	5.4152,	4.7135 / 2,	0.39828] # BSF is halved as the CWRU website displays the 2*BSF 
df.loc[df.signal_location == "FE", ['bpfo', 'bpfi', 'bsf', 'ftf']] = [3.0530,	4.9469,	3.9874 / 2,	0.38170]
df.loc[(df.signal_location == "BA") & (df.fault_location == "DE"), ['bpfo', 'bpfi', 'bsf', 'ftf']] = [3.5848,	5.4152,	4.7135,	0.39828]
df.loc[(df.signal_location == "BA") & (df.fault_location == "FE"), ['bpfo', 'bpfi', 'bsf', 'ftf']] = [3.0530,	4.9469,	3.9874,	0.38170]

# ---- Other adjustments ----
df['waveform_id'] = df.apply(lambda row: f"{row['fault_type']}_{row['fault_size']}_{row['fault_location']}_{row['signal_location']}_{int(row['fs']/1000)}_{row['load']}HP", axis=1)
df.loc[df.fault_size == 0, 'fs'] = 48000
df.loc[df.fault_size == 0, 'waveform_id'] = df.apply(lambda row: f"{row['fault_type']}_{row['signal_location']}_{int(row['fs']/1000)}_{row['load']}HP", axis=1)
df.load = df.load.astype(int)
df.rpm = df.rpm.astype(int)
df.fault_location = df.apply(lambda row: "Normal" if (row['fault_type'] == "Normal") else row['fault_location'], axis=1)

# Labels
df.loc[:, 'duration'] = df.apply(lambda row: len(row['signal']) / row['fs'], axis=1)
df.loc[:, 'inner'] = df.apply(lambda row: 1 if ("IR" in row.fault_type) & (row['fault_location'] == row['signal_location']) else 0, axis=1)
df.loc[:, 'outer'] = df.apply(lambda row: 1 if ("OR" in row.fault_type) & (row['fault_location'] == row['signal_location']) else 0, axis=1)
df.loc[:, 'ball'] = df.apply(lambda row: 1 if ("B" in row.fault_type) & (row['fault_location'] == row['signal_location']) else 0, axis=1)
df.loc[:, 'multiclass_label'] = df.inner + 2*df.outer + 3*df.ball

In [11]:
df.head()

,file,prefix,signal_name,rpm,signal,fault_metadata,fault_size,load,fault_type,fault_location,...,bpfo,bpfi,bsf,ftf,waveform_id,duration,inner,outer,ball,multiclass_label
0,110,X110,X110_DE_time,1772,"[0.03212676923076923, 0.07823076923076923, 0.1...",IR007_1_48k,7,1,IR,DE,...,3.5848,5.4152,2.35675,0.39828,IR_7_DE_DE_48_1HP,10.129667,1,0,0,1
1,110,X110,X110_FE_time,1772,"[0.08629090909090908, 0.11053454545454545, 0.1...",IR007_1_48k,7,1,IR,DE,...,3.0530,4.9469,1.99370,0.38170,IR_7_DE_FE_48_1HP,10.129667,0,0,0,0
2,119,X119,X119_DE_time,1772,"[-0.07212119760479042, 0.30310395209580837, 0....",B007_1,7,1,B,DE,...,3.5848,5.4152,2.35675,0.39828,B_7_DE_DE_12_1HP,10.117500,0,0,1,3
3,119,X119,X119_FE_time,1772,"[0.07745636363636363, 0.045610909090909084, -0...",B007_1,7,1,B,DE,...,3.0530,4.9469,1.99370,0.38170,B_7_DE_FE_12_1HP,10.117500,0,0,0,0
4,119,X119,X119_BA_time,1772,"[0.07327228486646885, 0.06932902077151336, 0.0...",B007_1,7,1,B,DE,...,3.5848,5.4152,4.71350,0.39828,B_7_DE_BA_12_1HP,10.117500,0,0,0,0


In [12]:
df[df.waveform_id.duplicated()] # There are a few duplicated waveform_ids, due to the fact that some files contain multiple signals;

,file,prefix,signal_name,rpm,signal,fault_metadata,fault_size,load,fault_type,fault_location,...,bpfo,bpfi,bsf,ftf,waveform_id,duration,inner,outer,ball,multiclass_label
47,175,X175,X175_DE_time,1772,"[-0.010016, -0.0029213333333333335, 0.00542533...",IR014_1_48k,14,1,IR,DE,...,3.5848,5.4152,2.35675,0.39828,IR_14_DE_DE_48_1HP,7.956042,1,0,0,1
79,99,X099,X099_DE_time,1750,"[0.06425353846153846, 0.06300184615384616, -0....",Normal_2,0,2,Normal,Normal,...,3.5848,5.4152,2.35675,0.39828,Normal_DE_48_2HP,10.105479,0,0,0,0
80,99,X099,X099_FE_time,1750,"[0.038625454545454546, 0.0967690909090909, 0.1...",Normal_2,0,2,Normal,Normal,...,3.0530,4.9469,1.99370,0.38170,Normal_FE_48_2HP,10.105479,0,0,0,0
148,217,X217,X217_DE_time,1730,"[-0.10096984615384616, -0.2561796923076923, -0...",IR021_3_48k,21,3,IR,DE,...,3.5848,5.4152,2.35675,0.39828,IR_21_DE_DE_48_3HP,10.190104,1,0,0,1
149,217,X217,X217_FE_time,1730,"[0.2342181818181818, 0.20134545454545455, 0.11...",IR021_3_48k,21,3,IR,DE,...,3.0530,4.9469,1.99370,0.38170,IR_21_DE_FE_48_3HP,10.190104,0,0,0,0


In [13]:
df[df.waveform_id == "IR_14_DE_DE_48_1HP"] # Example of duplicated waveform_id; Here, file 175 contains signal for possibly file 217. We will exclude those.

,file,prefix,signal_name,rpm,signal,fault_metadata,fault_size,load,fault_type,fault_location,...,bpfo,bpfi,bsf,ftf,waveform_id,duration,inner,outer,ball,multiclass_label
46,175,X217,X217_DE_time,1772,"[0.23782153846153845, 0.20444307692307692, 0.1...",IR014_1_48k,14,1,IR,DE,...,3.5848,5.4152,2.35675,0.39828,IR_14_DE_DE_48_1HP,10.190104,1,0,0,1
47,175,X175,X175_DE_time,1772,"[-0.010016, -0.0029213333333333335, 0.00542533...",IR014_1_48k,14,1,IR,DE,...,3.5848,5.4152,2.35675,0.39828,IR_14_DE_DE_48_1HP,7.956042,1,0,0,1


In [14]:
list_of_duplicated_ids = df[df.waveform_id.duplicated()].waveform_id.unique()
list_of_duplicated_ids

array(['IR_14_DE_DE_48_1HP', 'Normal_DE_48_2HP', 'Normal_FE_48_2HP',
       'IR_21_DE_DE_48_3HP', 'IR_21_DE_FE_48_3HP'], dtype=object)

In [15]:
# For each id, we will check if the prefix (e.g. X098) correspond to the actual file (e.g. 98.mat). If not, we will drop the duplicated rows.
for wid in list_of_duplicated_ids:
    subset = df[df.waveform_id == wid]
    indices_to_drop = []
    for idx, row in subset.iterrows():
        file = row['file']
        prefix = row.prefix.replace("X0", "").replace("X", "")
        if prefix != file:
            indices_to_drop.append(idx)
    df.drop(index=indices_to_drop, inplace=True)

In [16]:
final_cols = ['waveform_id', 'fault_location', 'load', 'rpm', 'fault_type',
       'fault_size', 'signal_location', 'fs', 'duration', 'bpfo', 'bpfi',
       'bsf', 'ftf', 'inner', 'outer', 'ball', 'multiclass_label', 'signal']
df = df[final_cols]

In [17]:
df

,waveform_id,fault_location,load,rpm,fault_type,fault_size,signal_location,fs,duration,bpfo,bpfi,bsf,ftf,inner,outer,ball,multiclass_label,signal
0,IR_7_DE_DE_48_1HP,DE,1,1772,IR,7,DE,48000,10.129667,3.5848,5.4152,2.35675,0.39828,1,0,0,1,"[0.03212676923076923, 0.07823076923076923, 0.1..."
1,IR_7_DE_FE_48_1HP,DE,1,1772,IR,7,FE,48000,10.129667,3.0530,4.9469,1.99370,0.38170,0,0,0,0,"[0.08629090909090908, 0.11053454545454545, 0.1..."
2,B_7_DE_DE_12_1HP,DE,1,1772,B,7,DE,12000,10.117500,3.5848,5.4152,2.35675,0.39828,0,0,1,3,"[-0.07212119760479042, 0.30310395209580837, 0...."
3,B_7_DE_FE_12_1HP,DE,1,1772,B,7,FE,12000,10.117500,3.0530,4.9469,1.99370,0.38170,0,0,0,0,"[0.07745636363636363, 0.045610909090909084, -0..."
4,B_7_DE_BA_12_1HP,DE,1,1772,B,7,BA,12000,10.117500,3.5848,5.4152,4.71350,0.39828,0,0,0,0,"[0.07327228486646885, 0.06932902077151336, 0.0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,IR_21_DE_DE_48_0HP,DE,0,1797,IR,21,DE,48000,5.090396,3.5848,5.4152,2.35675,0.39828,1,0,0,1,"[0.851568, 0.6287667692307692, 0.3158436923076..."
412,IR_21_DE_FE_48_0HP,DE,0,1797,IR,21,FE,48000,5.090396,3.0530,4.9469,1.99370,0.38170,0,0,0,0,"[0.8386654545454545, 0.61924, 0.31105818181818..."
413,B_21_FE_DE_12_1HP,FE,1,1772,B,21,DE,12000,10.112583,3.5848,5.4152,2.35675,0.39828,0,0,0,0,"[0.0320809381237525, 0.20141956087824353, 0.03..."
414,B_21_FE_FE_12_1HP,FE,1,1772,B,21,FE,12000,10.112583,3.0530,4.9469,1.99370,0.38170,0,0,1,3,"[-0.16300389610389612, 0.01625974025974026, 0...."


#### 1.3 Saving raw files (organized by their waveform_id) and CWRU metadata

In [18]:
for idx, row in df.iterrows():
    sig = pd.Series({"signal": [row.signal]})
    hp = row.load

    # create folder O HP, 1 HP, etc depending on the HP
    folder_name = os.path.join(dir_to_download, f"{hp} HP")
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)

    id = row.waveform_id.replace("_0HP","").replace("_1HP","").replace("_2HP","").replace("_3HP","")
    
    print(f"Saving {row.waveform_id} to {folder_name}/{id}.bz2")
    sig.to_pickle(os.path.join(folder_name, f"{id}.bz2"))

Saving IR_7_DE_DE_48_1HP to data/datasets/cwru/raw/1 HP/IR_7_DE_DE_48.bz2
Saving IR_7_DE_FE_48_1HP to data/datasets/cwru/raw/1 HP/IR_7_DE_FE_48.bz2
Saving B_7_DE_DE_12_1HP to data/datasets/cwru/raw/1 HP/B_7_DE_DE_12.bz2
Saving B_7_DE_FE_12_1HP to data/datasets/cwru/raw/1 HP/B_7_DE_FE_12.bz2
Saving B_7_DE_BA_12_1HP to data/datasets/cwru/raw/1 HP/B_7_DE_BA_12.bz2
Saving OR@6_7_DE_DE_12_2HP to data/datasets/cwru/raw/2 HP/OR@6_7_DE_DE_12.bz2
Saving OR@6_7_DE_FE_12_2HP to data/datasets/cwru/raw/2 HP/OR@6_7_DE_FE_12.bz2
Saving OR@6_7_DE_BA_12_2HP to data/datasets/cwru/raw/2 HP/OR@6_7_DE_BA_12.bz2
Saving OR@12_7_DE_DE_12_2HP to data/datasets/cwru/raw/2 HP/OR@12_7_DE_DE_12.bz2
Saving OR@12_7_DE_FE_12_2HP to data/datasets/cwru/raw/2 HP/OR@12_7_DE_FE_12.bz2
Saving OR@12_7_DE_BA_12_2HP to data/datasets/cwru/raw/2 HP/OR@12_7_DE_BA_12.bz2
Saving OR@12_7_DE_DE_12_3HP to data/datasets/cwru/raw/3 HP/OR@12_7_DE_DE_12.bz2
Saving OR@12_7_DE_FE_12_3HP to data/datasets/cwru/raw/3 HP/OR@12_7_DE_FE_12.bz2
Sa

In [19]:
df.drop(columns=["signal"]).to_pickle("data/metadata/cwru/files_metadata.bz2") # Saving metadata without the signal column

In [20]:
# If you want to delete all .mat files after processing, uncomment the following lines:
# for file in all_files:
#     os.remove(file)

#### 1.4 Reading CWRU signals based on metadata

In [21]:
from src.utils.add_signal_data import (add_signal_data_cwru)

In [22]:
df = pd.read_pickle("data/metadata/cwru/files_metadata.bz2")

In [23]:
#df_ex = df.sample(50) # Example dataframe with 50 samples without the signal column
df_with_signals = add_signal_data_cwru(df, fast_mode=False)

In [24]:
df_with_signals.to_pickle("data/datasets/cwru/processed/full_dataset.pkl")

In [25]:
df_with_signals = add_signal_data_cwru(df, fast_mode=True) # from 15.7s to 0.3s

#### 1.5 Saving filtered cwru metadata (which will be used in our experiments)

In [26]:
drive_fault_signals = (
    (df["fault_location"] == "DE")
    & (df["fs"] == 12000)
    & (df["fault_type"].isin(["B", "IR", "OR@6"]))
    & (df["signal_location"].isin(["DE", "FE"]))
    & (df["fault_size"].isin([7, 14, 21]))
)
fan_fault_signals = (
    (df["fault_location"] == "FE")
    & (df["signal_location"].isin(["DE", "FE"]))
    & (
        (
            (df["fault_size"].isin([14, 21]))
            & (df["load"].isin([1, 2, 3]))
            & (df["fault_type"].isin(["B", "IR", "OR@3"]))
        )
        | (df["fault_type"].isin(["B", "IR", "OR@6"]))
    )
)
normal_signals = (df["fault_type"] == "Normal") & (
    df["signal_location"].isin(["DE", "FE"])
)

In [27]:
filtered_cwru = (
    df[drive_fault_signals | fan_fault_signals].copy().reset_index(drop=True)
)

In [28]:
filtered_cwru.head()

,waveform_id,fault_location,load,rpm,fault_type,fault_size,signal_location,fs,duration,bpfo,bpfi,bsf,ftf,inner,outer,ball,multiclass_label
0,B_7_DE_DE_12_1HP,DE,1,1772,B,7,DE,12000,10.11750,3.5848,5.4152,2.35675,0.39828,0,0,1,3
1,B_7_DE_FE_12_1HP,DE,1,1772,B,7,FE,12000,10.11750,3.0530,4.9469,1.99370,0.38170,0,0,0,0
2,OR@6_7_DE_DE_12_2HP,DE,2,1750,OR@6,7,DE,12000,10.11750,3.5848,5.4152,2.35675,0.39828,0,1,0,2
3,OR@6_7_DE_FE_12_2HP,DE,2,1750,OR@6,7,FE,12000,10.11750,3.0530,4.9469,1.99370,0.38170,0,0,0,0
4,B_7_DE_DE_12_0HP,DE,0,1797,B,7,DE,12000,10.21425,3.5848,5.4152,2.35675,0.39828,0,0,1,3


In [29]:
filtered_cwru.to_pickle("data/splits/cwru/filtered_cwru.pkl")

# 2. PU dataset

In order to process the PU dataset, download all files from https://groups.uni-paderborn.de/kat/BearingDataCenter/ and unzip them on `data/datasets/paderborn/raw`.

The following function will process all files, save them in pickled .bz2 format and return a metadata organized list.

In [7]:
def process_matfiles(
    folder_path: Path, remove_signal: bool = True, save_signal: bool = False
) -> List[Dict]:
    """Function to process the mat files and return a list of dictionaries

    Used only once, not necessary to use it again.
    """
    output_list = []

    for _, filepath in enumerate(folder_path.rglob("*.mat")):
        # strip the folder path and get the filename only.
        key_name = str(filepath).split("/")[-1][:-4]
        print(filepath)

        file = loadmat(filepath)

        data = file[key_name]

        # Metadata from filename
        load_torque = 0.7 if key_name.split("_")[1] == "M07" else 0.1
        radial_force = 1000 if key_name.split("_")[2] == "F10" else 400
        rotational_speed = 1500 if key_name.split("_")[0] == "N15" else 900
        bearing_id = key_name.split("_")[-2]
        repetition = key_name.split("_")[-1]
        if "a" in bearing_id.lower():
            fault_label = "outer"
        elif "i" in bearing_id.lower():
            fault_label = "inner"
        elif "b" in bearing_id.lower():
            fault_label = "combination"
        else:
            fault_label = "normal"

        ids_severity_1 = [
            "KA01",
            "KA05",
            "KA07",
            "KI01",
            "KI03",
            "KI05",
            "KA04",
            "KA15",
            "KA22",
            "KA30",
            "KB27",
            "KI04",
            "KI14",
            "KI17",
            "KI21",
        ]
        ids_severity_2 = [
            "KA03",
            "KA06",
            "KA08",
            "KA09",
            "KI07",
            "KI08",
            "KA16",
            "KB23",
            "KI18",
        ]
        ids_severity_3 = ["KB24", "KI16"]

        if bearing_id in ids_severity_1:
            severity = 1
        elif bearing_id in ids_severity_2:
            severity = 2
        elif bearing_id in ids_severity_3:
            severity = 3
        else:
            severity = 0

        ids_artificial = [
            "KA01",
            "KA03",
            "KA05",
            "KA06",
            "KA07",
            "KA08",
            "KA09",
            "KI01",
            "KI03",
            "KI05",
            "KI07",
            "KI08",
        ]
        ids_real = [
            "KA04",
            "KA15",
            "KA16",
            "KA22",
            "KA30",
            "KB23",
            "KB24",
            "KB27",
            "KI04",
            "KI14",
            "KI16",
            "KI17",
            "KI18",
            "KI21",
        ]

        if bearing_id in ids_artificial:
            type_of_failure = "artificial"
        elif bearing_id in ids_real:
            type_of_failure = "real"
        else:
            type_of_failure = "normal"

        temp_dict = {key: data[key][0, 0] for key in data.dtype.names}
        # Equivalent to Description in temp_dict
        try:
            print(temp_dict["Description"]["General"][0, 0]["DateTime"])
            duration = temp_dict["Description"]["Measurement"][0, 0]["Length"][0][0][0][
                0
            ]
            date = temp_dict["Description"]["General"][0, 0]["DateTime"][0][0][0]
        except Exception as e:
            print(f"Error extracting duration and date: {e}")
            duration = None
            date = None

        data_dict = {
            # Equivalent to Info in temp_dict
            # "revision_major": temp_dict["Info"]["RevisionMajor"][0][0][0][0], # Unnecessary
            # "revision_minor": temp_dict["Info"]["RevisionMinor"][0][0][0][0], # Unnecessary
            # "measurement_id": temp_dict["Info"]["MeasurementID"][0][0][0][0], # Unnecessary
            # Metadata from filename
            "waveform_id": key_name,
            "bearing_id": bearing_id,
            "repetition": repetition,
            "load_torque": load_torque,
            "radial_force": radial_force,
            "rotational_speed": rotational_speed,
            "fault_label": fault_label,
            "severity": severity,
            "type_of_failure": type_of_failure,
            # Equivalent to X in temp_dict
            # "time_1Hz": temp_dict["X"][0, 2]['Data'][0], # Unnecessary
            # "time_4kHz": temp_dict["X"][0, 0]['Data'][0], # Unnecessary
            # "time_64kHz": temp_dict["X"][0, 1]['Data'][0], # Unnecessary
            # Equivalent to Y in temp_dict
            "force": temp_dict["Y"][0, 0]["Data"][0],
            "phase_current_1": temp_dict["Y"][0, 1]["Data"][0],
            "phase_current_2": temp_dict["Y"][0, 2]["Data"][0],
            "speed": temp_dict["Y"][0, 3]["Data"][0],
            "temperature": temp_dict["Y"][0, 4]["Data"][0],
            "torque": temp_dict["Y"][0, 5]["Data"][0],
            "vibration": temp_dict["Y"][0, 6]["Data"][0],
            # Equivalent to Description in temp_dict
            "duration": duration,
            "date": date,
            # Extra information
            "fs_force": "4000",
            "fs_phase_current": "64000",
            "fs_speed": "4000",
            "fs_temperature": "1",
            "fs_torque": "4000",
            "fs_vibration": "64000",
        }

        if save_signal:
            save_path = str(filepath)[:-4] + ".bz2"
            temp_save_dict = {}
            signal_keys = [
                "force",
                "phase_current_1",
                "phase_current_2",
                "speed",
                "temperature",
                "torque",
                "vibration",
                "fs_force",
                "fs_phase_current",
                "fs_speed",
                "fs_temperature",
                "fs_torque",
                "fs_vibration",
            ]
            for key in signal_keys:
                temp_save_dict[key] = [data_dict[key]]

            pd.DataFrame(temp_save_dict).to_pickle(save_path)

        if remove_signal:
            data_dict.pop("force")
            data_dict.pop("phase_current_1")
            data_dict.pop("phase_current_2")
            data_dict.pop("speed")
            data_dict.pop("temperature")
            data_dict.pop("torque")
            data_dict.pop("vibration")
            data_dict.pop("fs_force")
            data_dict.pop("fs_phase_current")
            data_dict.pop("fs_speed")
            data_dict.pop("fs_temperature")
            data_dict.pop("fs_torque")
            data_dict.pop("fs_vibration")

        output_list.append(data_dict)

    return output_list

In [8]:
folder_path = Path("data/datasets/paderborn/raw")
output = process_matfiles(folder_path, remove_signal=False, save_signal=False)

data/datasets/paderborn/raw/KA01/N15_M07_F10_KA01_3.mat
[[array(['10.11.2014 10:24:46'], dtype='<U19')]]
data/datasets/paderborn/raw/KA01/N15_M01_F10_KA01_10.mat
[[array(['10.11.2014 11:26:18'], dtype='<U19')]]
data/datasets/paderborn/raw/KA01/N15_M07_F10_KA01_5.mat
[[array(['10.11.2014 10:25:57'], dtype='<U19')]]
data/datasets/paderborn/raw/KA01/N15_M07_F04_KA01_11.mat
[[array(['10.11.2014 11:45:30'], dtype='<U19')]]
data/datasets/paderborn/raw/KA01/N15_M07_F10_KA01_15.mat
[[array(['10.11.2014 10:33:12'], dtype='<U19')]]
data/datasets/paderborn/raw/KA01/N15_M07_F04_KA01_16.mat
[[array(['10.11.2014 11:48:04'], dtype='<U19')]]
data/datasets/paderborn/raw/KA01/N15_M07_F10_KA01_12.mat
[[array(['10.11.2014 10:30:35'], dtype='<U19')]]
data/datasets/paderborn/raw/KA01/N15_M01_F10_KA01_17.mat
[[array(['10.11.2014 11:33:26'], dtype='<U19')]]
data/datasets/paderborn/raw/KA01/N09_M07_F10_KA01_6.mat
[[array(['10.11.2014 10:55:18'], dtype='<U19')]]
data/datasets/paderborn/raw/KA01/N15_M07_F04_KA01

In [9]:
df = pd.DataFrame(output)

In [10]:
df.head(1)

,waveform_id,bearing_id,repetition,load_torque,radial_force,rotational_speed,fault_label,severity,type_of_failure,force,...,torque,vibration,duration,date,fs_force,fs_phase_current,fs_speed,fs_temperature,fs_torque,fs_vibration
0,N15_M07_F10_KA01_3,KA01,3,0.7,1000,1500,outer,1,artificial,"[1100.2974601860403, 1105.1802726828134, 1106....",...,"[1.4286729598827435, 1.4284169677191825, 1.408...","[-0.10375976555642881, -0.051879882778214405, ...",4.0,10.11.2014 10:24:46,4000,64000,4000,1,4000,64000


In [11]:
manufacturers = {
    "FAG": ['KB27', 'KI16', 'KB24', 'KA04', 'KA15', 'KI21'],
     "IBU": ['KA08', 'K005', 'KA07', 'KA05', 'K002', 'K001', 'KA06', 'KI07',
       'KA22', 'K006', 'K003', 'K004', 'KA09', 'KI08', 'KI05'],
      "MTK": ['KI14', 'KI18', 'KA16', 'KA30', 'KI01', 'KA01', 'KI17', 'KA03',
       'KI04', 'KI03', 'KB23']
}

In [12]:
dim1 = ['KA08', 'KI14', 'KI18', 'K005', 'KA16', 'KA07', 'KA05', 'K002',
       'KA30', 'K001', 'KI01', 'KA06', 'KI07', 'KA22', 'KA01', 'KI17',
       'K006', 'KA03', 'K003', 'KI04', 'K004', 'KI03', 'KA09', 'KI08',
       'KI05', 'KB23']

dim2 = ['KB27', 'KI16', 'KB24', 'KA04', 'KA15', 'KI21']

bpfo = {
    3.0704: dim1,
      3.0548: dim2
}

bpfi = {
    4.9296: dim1,
    4.9452: dim2,
}

bsf = {
    2.036: dim1,
    1.996: dim2,
}

ftf = {
    0.3838: dim1,
    0.3819: dim2,
}

In [13]:
df["bearing_manufacturer"] = df.bearing_id.apply(
    lambda x: next((manu for manu, ids in manufacturers.items() if x in ids), None)
)
df["bpfo"] = df.bearing_id.apply(
    lambda x: next((freq for freq, ids in bpfo.items() if x in ids), None)
)
df["bpfi"] = df.bearing_id.apply(
    lambda x: next((freq for freq, ids in bpfi.items() if x in ids), None)
)
df["bsf"] = df.bearing_id.apply(
    lambda x: next((freq for freq, ids in bsf.items() if x in ids), None)
)
df["ftf"] = df.bearing_id.apply(
    lambda x: next((freq for freq, ids in ftf.items() if x in ids), None)
)

In [14]:
df["inner"] = df.apply(lambda row: 1 if (row['fault_label'] == "inner") or (row['fault_label'] == "combination") else 0, axis=1)
df["outer"] = df.apply(lambda row: 1 if (row['fault_label'] == "outer") or (row['fault_label'] == "combination") else 0, axis=1)
df["ball"] = 0
df["cage"] = 0
df["multiclass_label"] = df.inner + 2*df.outer # this excludes combined as an extra class. since we will only work with multilabel, this will not be necessary. 

In [15]:
df['fs'] = 64000

In [16]:
df['duration'] = df.vibration.apply(lambda x: len(x)/64000)

In [17]:
df['date'] = pd.to_datetime(df['date'])

In [18]:
cols = ['waveform_id', 'bearing_id', 'bearing_manufacturer', 'repetition',
       'load_torque', 'radial_force', 'rotational_speed', 'fault_label',
       'severity', 'type_of_failure', 'duration', 'date', 'bpfo', 'bpfi',
       'bsf', 'ftf', 'multiclass_label', 'inner', 'outer', 'ball', 'cage', 'fs', 'vibration'
       ]

df = df[cols]

In [19]:
df[df.waveform_id.duplicated()].shape

(80, 23)

In [20]:
df.drop_duplicates(subset=['waveform_id'], inplace=True)

In [21]:
df.head()

,waveform_id,bearing_id,bearing_manufacturer,repetition,load_torque,radial_force,rotational_speed,fault_label,severity,type_of_failure,...,bpfi,bsf,ftf,multiclass_label,inner,outer,ball,cage,fs,vibration
0,N15_M07_F10_KA01_3,KA01,MTK,3,0.7,1000,1500,outer,1,artificial,...,4.9296,2.036,0.3838,2,0,1,0,0,64000,"[-0.10375976555642881, -0.051879882778214405, ..."
1,N15_M01_F10_KA01_10,KA01,MTK,10,0.1,1000,1500,outer,1,artificial,...,4.9296,2.036,0.3838,2,0,1,0,0,64000,"[0.2716064451330048, 0.0244140624838656, -0.21..."
2,N15_M07_F10_KA01_5,KA01,MTK,5,0.7,1000,1500,outer,1,artificial,...,4.9296,2.036,0.3838,2,0,1,0,0,64000,"[0.7720947260522496, 0.3936767575523328, 0.189..."
3,N15_M07_F04_KA01_11,KA01,MTK,11,0.7,400,1500,outer,1,artificial,...,4.9296,2.036,0.3838,2,0,1,0,0,64000,"[-0.1586914061451264, -0.0946044921249792, 0.0..."
4,N15_M07_F10_KA01_15,KA01,MTK,15,0.7,1000,1500,outer,1,artificial,...,4.9296,2.036,0.3838,2,0,1,0,0,64000,"[-0.1739501951975424, -0.1861572264394752, -0...."


In [ ]:
_ = process_matfiles(folder_path, remove_signal=False, save_signal=True) # Run this to save all .mat files in .bz2 files ~12 minutes

In [ ]:
# Run this if you want to remove original .mat files after processing
# for _, filepath in enumerate(folder_path.rglob("*.mat")):
#    os.remove(filepath)

**EXAMPLE**: loading PU signals using metadata df

In [22]:
from src.utils.add_signal_data import (add_signal_data_paderborn)

In [23]:
add_signal_data_paderborn(df.sample(10), only_vibration=True, fast_mode=False)

,waveform_id,bearing_id,bearing_manufacturer,repetition,load_torque,radial_force,rotational_speed,fault_label,severity,type_of_failure,...,bsf,ftf,multiclass_label,inner,outer,ball,cage,fs,vibration_x,vibration_y
0,N15_M07_F10_KI21_7,KI21,FAG,7,0.7,1000,1500,inner,1,real,...,1.996,0.3819,1,1,0,0,0,64000,"[-0.0976562499354624, -0.0640869140201472, -0....","[-0.0976562499354624, -0.0640869140201472, -0...."
1,N09_M07_F10_KI07_7,KI07,IBU,7,0.7,1000,900,inner,2,artificial,...,2.036,0.3838,1,1,0,0,0,64000,"[-0.1464843749031936, -0.0549316405886976, -0....","[-0.1464843749031936, -0.0549316405886976, -0...."
2,N15_M07_F04_KA07_2,KA07,IBU,2,0.7,400,1500,outer,1,artificial,...,2.036,0.3838,2,0,1,0,0,64000,"[-0.0061035156209664, -0.091552734314496, -0.2...","[-0.0061035156209664, -0.091552734314496, -0.2..."
3,N15_M01_F10_KI21_5,KI21,FAG,5,0.1,1000,1500,inner,1,real,...,1.996,0.3819,1,1,0,0,0,64000,"[-0.1281738280402944, -0.4974365231087616, -0....","[-0.1281738280402944, -0.4974365231087616, -0...."
4,N09_M07_F10_KA09_15,KA09,IBU,15,0.7,1000,900,outer,2,artificial,...,2.036,0.3838,2,0,1,0,0,64000,"[-0.07629394526208, 0.057983398399180805, 0.07...","[-0.07629394526208, 0.057983398399180805, 0.07..."
5,N15_M07_F10_KB23_16,KB23,MTK,16,0.7,1000,1500,combination,2,real,...,2.036,0.3838,3,1,1,0,0,64000,"[-0.0091552734314496, 0.1892089842499584, 0.17...","[-0.0091552734314496, 0.1892089842499584, 0.17..."
6,N15_M07_F04_KA06_15,KA06,IBU,15,0.7,400,1500,outer,2,artificial,...,2.036,0.3838,2,0,1,0,0,64000,"[0.5737304683708416, 1.1413574211207167, 1.040...","[0.5737304683708416, 1.1413574211207167, 1.040..."
7,N15_M07_F10_KA22_6,KA22,IBU,6,0.7,1000,1500,outer,1,real,...,2.036,0.3838,2,0,1,0,0,64000,"[-0.20751953111285762, -0.21362304673382398, -...","[-0.20751953111285762, -0.21362304673382398, -..."
8,N09_M07_F10_KA08_14,KA08,IBU,14,0.7,1000,900,outer,2,artificial,...,2.036,0.3838,2,0,1,0,0,64000,"[0.0396728515362816, -0.0244140624838656, 0.05...","[0.0396728515362816, -0.0244140624838656, 0.05..."
9,N15_M07_F04_KI16_16,KI16,FAG,16,0.7,400,1500,inner,3,real,...,1.996,0.3819,1,1,0,0,0,64000,"[-0.19226074206044158, -0.0183105468628992, -0...","[-0.19226074206044158, -0.0183105468628992, -0..."


**IMPORTANT**: saving full dataframe (including signals) for fast loading

In [52]:
df.to_pickle("data/datasets/paderborn/processed/vibration_only/full_dataset.pkl")

In [24]:
add_signal_data_paderborn(df, only_vibration=True, fast_mode=True) # 3.5s to open full dataset

,waveform_id,bearing_id,bearing_manufacturer,repetition,load_torque,radial_force,rotational_speed,fault_label,severity,type_of_failure,...,bsf,ftf,multiclass_label,inner,outer,ball,cage,fs,vibration_x,vibration_y
0,N15_M07_F10_KA01_3,KA01,MTK,3,0.7,1000,1500,outer,1,artificial,...,2.036,0.3838,2,0,1,0,0,64000,"[-0.10375976555642881, -0.051879882778214405, ...","[-0.10375976555642881, -0.051879882778214405, ..."
1,N15_M01_F10_KA01_10,KA01,MTK,10,0.1,1000,1500,outer,1,artificial,...,2.036,0.3838,2,0,1,0,0,64000,"[0.2716064451330048, 0.0244140624838656, -0.21...","[0.2716064451330048, 0.0244140624838656, -0.21..."
2,N15_M07_F10_KA01_5,KA01,MTK,5,0.7,1000,1500,outer,1,artificial,...,2.036,0.3838,2,0,1,0,0,64000,"[0.7720947260522496, 0.3936767575523328, 0.189...","[0.7720947260522496, 0.3936767575523328, 0.189..."
3,N15_M07_F04_KA01_11,KA01,MTK,11,0.7,400,1500,outer,1,artificial,...,2.036,0.3838,2,0,1,0,0,64000,"[-0.1586914061451264, -0.0946044921249792, 0.0...","[-0.1586914061451264, -0.0946044921249792, 0.0..."
4,N15_M07_F10_KA01_15,KA01,MTK,15,0.7,1000,1500,outer,1,artificial,...,2.036,0.3838,2,0,1,0,0,64000,"[-0.1739501951975424, -0.1861572264394752, -0....","[-0.1739501951975424, -0.1861572264394752, -0...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2555,N15_M07_F04_KA04_11,KA04,FAG,11,0.7,400,1500,outer,1,real,...,1.996,0.3819,2,0,1,0,0,64000,"[0.46691894500392955, 0.3692626950684672, 0.55...","[0.46691894500392955, 0.3692626950684672, 0.55..."
2556,N09_M07_F10_KA04_8,KA04,FAG,8,0.7,1000,900,outer,1,real,...,1.996,0.3819,2,0,1,0,0,64000,"[-0.0366210937257984, -0.0427246093467648, 0.0...","[-0.0366210937257984, -0.0427246093467648, 0.0..."
2557,N09_M07_F10_KA04_2,KA04,FAG,2,0.7,1000,900,outer,1,real,...,1.996,0.3819,2,0,1,0,0,64000,"[-0.3112792966692864, -0.0976562499354624, 0.0...","[-0.3112792966692864, -0.0976562499354624, 0.0..."
2558,N15_M01_F10_KA04_19,KA04,FAG,19,0.1,1000,1500,outer,1,real,...,1.996,0.3819,2,0,1,0,0,64000,"[-0.20446777330237442, 0.061035156209664, 0.16...","[-0.20446777330237442, 0.061035156209664, 0.16..."


In [21]:
df.drop(columns=["vibration"]).to_pickle("data/metadata/paderborn/files_metadata.bz2") # Saving metadata without the signal column

# 3. UORED-VAFLCS (Ottawa)

In order to process UORED-VAFCLS dataset, download folder `1_CSV_Raw_Data_Files (.csv)` from https://data.mendeley.com/datasets/y2px5tg92h/5 and paste the subfolders on `data/datasets/ottawa/raw`.

In [ ]:
ott_folder = "data/datasets/ottawa/raw" 
data = []

for _, filepath in enumerate(Path(ott_folder).rglob("*.csv")):
    print(filepath)
    df_temp = pd.read_csv(filepath)
    vibration = df_temp.Accelerometer.values # we are only processing vibration in this paper.
    rpm = df_temp.Speed.iloc[0]
    load = df_temp.Load.iloc[0]
    path = str(filepath).split("/") # e.g. "IR", "OR", "B", "Normal"
    fault_type = path[-2].split("_")[1]
    severity = path[-1].strip(".csv")[-1]
    bearing_id = path[-1].strip(".csv").split("_")[1]
    waveform_id = f"{bearing_id}_{fault_type}_{severity}"
    data.append({
        "waveform_id": waveform_id,
        "bearing_id": bearing_id,
        "fault_type": fault_type,
        "severity": int(severity),
        "load": load,
        "rpm": rpm,
        "fs": 42000,
        "vibration": vibration,
        "duration": 10.0
    })


In [26]:
df = pd.DataFrame(data)

In [27]:
df.head()

,waveform_id,bearing_id,fault_type,severity,load,rpm,fs,vibration,duration
0,16_Healthy_0,16,Healthy,0,400,1780,42000,"[-7.696933, -7.632468, -7.664701, -7.858096, -...",10.0
1,2_Healthy_0,2,Healthy,0,400,1800,42000,"[193.402454, 192.7578, 191.726353, 190.662674,...",10.0
2,17_Healthy_0,17,Healthy,0,400,1800,42000,"[85.745358, 85.680893, 85.584195, 85.423032, 8...",10.0
3,8_Healthy_0,8,Healthy,0,400,1948,42000,"[-0.895861, -0.831396, -0.347907, -0.218977, 0...",10.0
4,1_Healthy_0,1,Healthy,0,400,1819,42000,"[-0.186744, 0.844698, 2.069536, 3.358839, 4.45...",10.0


In [28]:
bpfo = 3.05; bpfi = 4.95; bsf = 1.986; ftf = 0.381 # Bearing characteristic frequencies calculated based on original UORED-VAFCLS paper info

df["bpfo"] = bpfo
df["bpfi"] = bpfi
df["bsf"] = bsf
df["ftf"] = ftf

df.loc[:, "severity"] = df.severity.astype(int)

In [29]:
df['inner'] = df.apply(lambda row: 1 if row['fault_type'] == "Inner" else 0, axis=1)
df['outer'] = df.apply(lambda row: 1 if row['fault_type'] == "Outer" else 0, axis=1)
df['ball'] = df.apply(lambda row: 1 if row['fault_type'] == "Ball" else 0, axis=1)
df['cage'] = df.apply(lambda row: 1 if row['fault_type'] == "Cage" else 0, axis=1)

In [30]:
df.head(10)

,waveform_id,bearing_id,fault_type,severity,load,rpm,fs,vibration,duration,bpfo,bpfi,bsf,ftf,inner,outer,ball,cage
0,16_Healthy_0,16,Healthy,0,400,1780,42000,"[-7.696933, -7.632468, -7.664701, -7.858096, -...",10.0,3.05,4.95,1.986,0.381,0,0,0,0
1,2_Healthy_0,2,Healthy,0,400,1800,42000,"[193.402454, 192.7578, 191.726353, 190.662674,...",10.0,3.05,4.95,1.986,0.381,0,0,0,0
2,17_Healthy_0,17,Healthy,0,400,1800,42000,"[85.745358, 85.680893, 85.584195, 85.423032, 8...",10.0,3.05,4.95,1.986,0.381,0,0,0,0
3,8_Healthy_0,8,Healthy,0,400,1948,42000,"[-0.895861, -0.831396, -0.347907, -0.218977, 0...",10.0,3.05,4.95,1.986,0.381,0,0,0,0
4,1_Healthy_0,1,Healthy,0,400,1819,42000,"[-0.186744, 0.844698, 2.069536, 3.358839, 4.45...",10.0,3.05,4.95,1.986,0.381,0,0,0,0
5,5_Healthy_0,5,Healthy,0,400,1799,42000,"[-6.053072, -10.92019, -12.04833, -11.758237, ...",10.0,3.05,4.95,1.986,0.381,0,0,0,0
6,15_Healthy_0,15,Healthy,0,400,1720,42000,"[3.391071, 4.035723, 2.972048, 3.004281, 2.295...",10.0,3.05,4.95,1.986,0.381,0,0,0,0
7,19_Healthy_0,19,Healthy,0,400,1785,42000,"[55.575632, 56.316982, 56.542611, 56.768239, 5...",10.0,3.05,4.95,1.986,0.381,0,0,0,0
8,9_Healthy_0,9,Healthy,0,400,1800,42000,"[-0.412372, -1.185954, -3.700095, -4.667072, -...",10.0,3.05,4.95,1.986,0.381,0,0,0,0
9,11_Healthy_0,11,Healthy,0,400,1709,42000,"[-9.566422, -9.501957, -8.792841, -4.119118, -...",10.0,3.05,4.95,1.986,0.381,0,0,0,0


In [ ]:
df.drop(columns=["vibration"]).to_pickle("data/metadata/ottawa/files_metadata.bz2") # Saving metadata without the signal column

In [9]:
df.to_pickle("data/datasets/ottawa/processed/vibration_only/full_dataset.pkl") # Saving full dataset with signals for fast load when training